In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset

print("Preparing Dataset...")
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)

pretrain_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

tasks = []
pairs = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]

for (c0, c1) in pairs:
    train_mask = (train_dataset.targets == c0) | (train_dataset.targets == c1)
    X_train = train_dataset.data[train_mask].float() / 255.0
    X_train = (X_train - 0.1307) / 0.3081
    X_train = X_train.view(-1, 1, 28, 28)
    y_train = (train_dataset.targets[train_mask] == c1).float().view(-1, 1)
    
    test_mask = (test_dataset.targets == c0) | (test_dataset.targets == c1)
    X_test = test_dataset.data[test_mask].float() / 255.0
    X_test = (X_test - 0.1307) / 0.3081
    X_test = X_test.view(-1, 1, 28, 28)
    y_test = (test_dataset.targets[test_mask] == c1).float().view(-1, 1)
    
    tasks.append((f"Digits {c0} vs {c1}", X_train, y_train, X_test, y_test))



In [ ]:

class PyTorchDynamicNetwork(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, max_neurons: int = 2000, steps: int = 3):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.max_neurons = max_neurons
        self.steps = steps

        # Generation 2: Output indices track which neurons are output heads
        self.output_indices = list(range(input_dim, input_dim + output_dim))
        
        # Start with input + output + 4 hidden neurons
        self.active_neurons = input_dim + output_dim + 4

        # Adjacency Matrix (Structure)
        self.register_buffer('M', torch.zeros(max_neurons, max_neurons))
        # Trainable Mask (Memory Protection)
        self.register_buffer('trainable_mask', torch.ones(max_neurons, max_neurons))
        
        # Generation 2: Localized Per-Neuron Stress
        self.register_buffer('neuron_stress', torch.zeros(max_neurons))

        # Weight Matrix and Bias
        self.W = nn.Parameter(torch.zeros(max_neurons, max_neurons))
        self.b = nn.Parameter(torch.zeros(max_neurons))

        self._hook_registered = False
        self._init_random_connections()

    def _init_random_connections(self):
        with torch.no_grad():
            for i in range(self.active_neurons):
                for j in range(self.active_neurons):
                    # Neurons can't connect to themselves, and inputs receive no connections
                    if i != j and i >= self.input_dim:
                        if torch.rand(1).item() > 0.5:
                            self.M[i, j] = 1.0
            
            # Initialize weights where connections exist
            mask = self.M[:self.active_neurons, :self.active_neurons].bool()
            self.W[:self.active_neurons, :self.active_neurons][mask] = torch.randn(mask.sum()) * 0.1

    def _register_hooks(self):
        def _w_hook(grad):
            return grad * self.trainable_mask * self.M

        def _b_hook(grad):
            # If a neuron's incoming connections are ALL frozen, freeze its bias too
            b_mask = torch.ones_like(grad)
            for i in range(self.active_neurons):
                # If it has incoming connections but ALL of them are frozen (trainable=0)
                if self.M[i, :].sum() > 0 and (self.M[i, :] * self.trainable_mask[i, :]).sum() == 0:
                    b_mask[i] = 0.0
            return grad * b_mask

        self.W.register_hook(_w_hook)
        self.b.register_hook(_b_hook)
        self._hook_registered = True

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self._hook_registered:
            self._register_hooks()

        batch = x.size(0)
        state = torch.zeros(batch, self.max_neurons, device=x.device)
        state[:, :self.input_dim] = x

        W_eff = self.W * self.M
        for _ in range(self.steps):
            new_state = torch.relu(torch.matmul(state, W_eff.T) + self.b)
            # Only update non-input neurons
            state = torch.cat([x, new_state[:, self.input_dim:]], dim=1)

        # Generation 2: Return output based on flexible indices
        return state[:, self.output_indices]

    def update_stress(self, grads_task: torch.Tensor, grads_replay: torch.Tensor, beta: float = 0.9):
        """Generation 2: Computes stress PER NEURON based on incoming gradient conflict."""
        with torch.no_grad():
            # conflict > 0 means gradients are pushing in opposite directions
            # CRITICAL: We must clamp this to >= 0 so stress doesn't become negative when gradients align!
            conflict = torch.relu(-(grads_task * grads_replay))
            
            # Only consider active, trainable connections
            active_mask = self.M * self.trainable_mask
            
            # Sum the conflict for all incoming connections to a neuron
            conflict_sum = (conflict * active_mask).sum(dim=1)
            
            # Count how many active, trainable connections each neuron has
            num_incoming = active_mask.sum(dim=1)
            
            # Local stress is the average conflict per incoming connection
            local_stress = conflict_sum / (num_incoming + 1e-8)
            
            # Update Exponential Moving Average
            self.neuron_stress = beta * self.neuron_stress + (1 - beta) * local_stress

    def get_max_neuron_stress(self) -> float:
        """Returns the highest stress level among all active neurons."""
        if self.active_neurons == 0: return 0.0
        return self.neuron_stress[:self.active_neurons].max().item()

    def stress_freeze(self, threshold: float = 0.3) -> int:
        """Generation 2: Freezes ONLY the neurons whose local stress exceeds the threshold."""
        n_frozen = 0
        with torch.no_grad():
            stressed_neurons = (self.neuron_stress > threshold) & (torch.arange(self.max_neurons, device=self.neuron_stress.device) < self.active_neurons)
            
            for i in torch.where(stressed_neurons)[0]:
                i = i.item()
                # Find its incoming connections that are currently trainable
                incoming = (self.trainable_mask[i, :] == 1) & (self.M[i, :] == 1)
                num_to_freeze = incoming.sum().item()
                
                if num_to_freeze > 0:
                    self.trainable_mask[i, incoming] = 0.0
                    n_frozen += num_to_freeze
                    # Reset stress since it is now protected
                    self.neuron_stress[i] = 0.0
                    
        return n_frozen

    def grow_neuron(self, num_connections: int = 15):
        """Grows a single hidden neuron and wires it up."""
        if self.active_neurons >= self.max_neurons:
            return -1
            
        new_idx = self.active_neurons
        self.active_neurons += 1
        
        with torch.no_grad():
            self.W[new_idx, :] = 0.0
            
            # 1. Incoming connections: from inputs and other hidden neurons
            valid_sources = list(range(self.input_dim)) + [i for i in range(new_idx) if i not in self.output_indices]
            if len(valid_sources) > 0:
                k = min(num_connections, len(valid_sources))
                chosen = torch.tensor(valid_sources)[torch.randperm(len(valid_sources))[:k]]
                self.M[new_idx, chosen] = 1.0
                self.W[new_idx, chosen] = torch.randn(k, device=self.W.device) * 0.1
                
            # 2. Outgoing connections: attach it to ALL current output heads so it's useful immediately!
            for out_idx in self.output_indices:
                self.M[out_idx, new_idx] = 1.0
                self.W[out_idx, new_idx] = torch.randn(1, device=self.W.device).item() * 0.1
                
        return new_idx

    def grow_output_head(self, num_connections: int = 15):
        """Generation 2: Dynamically spawns a brand new Output Neuron (Multi-Head)."""
        if self.active_neurons >= self.max_neurons:
            return -1
            
        new_idx = self.active_neurons
        self.active_neurons += 1
        self.output_dim += 1
        self.output_indices.append(new_idx)
        
        with torch.no_grad():
            self.W[new_idx, :] = 0.0
            
            # Incoming connections: from hidden neurons only (or inputs)
            hidden_and_input = list(range(self.input_dim)) + [i for i in range(new_idx) if i not in self.output_indices[:-1]]
            
            if len(hidden_and_input) > 0:
                k = min(num_connections, len(hidden_and_input))
                chosen = torch.tensor(hidden_and_input)[torch.randperm(len(hidden_and_input))[:k]]
                self.M[new_idx, chosen] = 1.0
                self.W[new_idx, chosen] = torch.randn(k, device=self.W.device) * 0.1
                
        return new_idx

    def n_trainable(self) -> int:
        return int((self.M[:self.active_neurons, :self.active_neurons] * self.trainable_mask[:self.active_neurons, :self.active_neurons]).sum().item())
        
    def n_frozen(self) -> int:
        return int((self.M[:self.active_neurons, :self.active_neurons] * (1 - self.trainable_mask[:self.active_neurons, :self.active_neurons])).sum().item())



In [ ]:
class CNNFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(32 * 7 * 7, 64)
        self.relu3 = nn.ReLU()

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.flatten(x)
        x = self.relu3(self.fc(x))
        return x

class HybridModel(nn.Module):
    def __init__(self, cnn, device):
        super().__init__()
        self.cnn = cnn
        # Gen 2: We start with output_dim=1 for the first task
        self.dynamic = PyTorchDynamicNetwork(input_dim=64, output_dim=1, max_neurons=500).to(device)
        
    def forward(self, x):
        features = self.cnn(x)
        # Returns [batch, num_heads]
        return self.dynamic(features)

class ReplayBuffer:
    def __init__(self, capacity: int = 500):
        self.capacity = capacity
        self.X = None
        self.y = None
        self.task_ids = None

    def add_data(self, X_new: torch.Tensor, y_new: torch.Tensor, task_id: int, num_samples: int = 100):
        idx = torch.randperm(X_new.size(0))[:num_samples]
        X_sub = X_new[idx].clone().detach()
        y_sub = y_new[idx].clone().detach()
        t_sub = torch.full((num_samples,), task_id, dtype=torch.long)

        if self.X is None:
            self.X = X_sub
            self.y = y_sub
            self.task_ids = t_sub
        else:
            self.X = torch.cat([self.X, X_sub], dim=0)
            self.y = torch.cat([self.y, y_sub], dim=0)
            self.task_ids = torch.cat([self.task_ids, t_sub], dim=0)

            if self.X.size(0) > self.capacity:
                keep = torch.randperm(self.X.size(0))[:self.capacity]
                self.X = self.X[keep]
                self.y = self.y[keep]
                self.task_ids = self.task_ids[keep]

    def sample(self, batch_size: int = 64) -> tuple:
        if self.X is None:
            return None, None, None
        size = self.X.size(0)
        idx = torch.randint(0, size, (min(batch_size, size),))
        return self.X[idx], self.y[idx], self.task_ids[idx]

    def has_data(self) -> bool:
        return self.X is not None and self.X.size(0) > 0

def _reset_adam_for_neuron(optimizer, net, new_idx):
    if net.W in optimizer.state:
        s_W = optimizer.state[net.W]
        if 'exp_avg' in s_W:
            s_W['exp_avg'][new_idx, :] = 0.0
            s_W['exp_avg'][:, new_idx] = 0.0
            s_W['exp_avg_sq'][new_idx, :] = 0.0
            s_W['exp_avg_sq'][:, new_idx] = 0.0
            
    if hasattr(net, 'b') and net.b in optimizer.state:
        s_b = optimizer.state[net.b]
        if 'exp_avg' in s_b:
            s_b['exp_avg'][new_idx] = 0.0
            s_b['exp_avg_sq'][new_idx] = 0.0



In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on {device}...")

print("\n" + "="*50)
print("--- Pre-Training CNN Feature Extractor ---")
print("="*50)

shared_cnn = CNNFeatureExtractor().to(device)
classifier = nn.Linear(64, 10).to(device)
optimizer_cnn = torch.optim.Adam(list(shared_cnn.parameters()) + list(classifier.parameters()), lr=1e-3)
criterion_cnn = nn.CrossEntropyLoss()

shared_cnn.train()
for epoch in range(2):
    for bx, by in pretrain_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer_cnn.zero_grad()
        out = classifier(shared_cnn(bx))
        loss = criterion_cnn(out, by)
        loss.backward()
        optimizer_cnn.step()
    print(f"Pre-Train Epoch {epoch+1} finished.")

for param in shared_cnn.parameters():
    param.requires_grad = False
shared_cnn.eval()
print("CNN Frozen.")



In [ ]:
print("\n" + "="*50)
print("--- Training Generation 2 (Multi-Head & Local Stress) ---")
print("="*50)

hybrid_model = HybridModel(shared_cnn, device)
replay = ReplayBuffer(capacity=1000)
optimizer = torch.optim.Adam(hybrid_model.dynamic.parameters(), lr=1e-3)
criterion = nn.MSELoss()

dynamic_accuracies = {}

for task_id, (name, X_train, y_train, X_test, y_test) in enumerate(tasks):
    print(f"\nTraining Task {task_id}: {name}...")
    
    # Generation 2: Multi-Head Output Expansion!
    if task_id > 0:
        new_out_idx = hybrid_model.dynamic.grow_output_head(num_connections=20)
        _reset_adam_for_neuron(optimizer, hybrid_model.dynamic, new_out_idx)
        print(f"  [Task Boundary] Spawned new Output Head #{task_id}. active={hybrid_model.dynamic.active_neurons}")
    
    dataset = TensorDataset(X_train, y_train)
    loader = DataLoader(dataset, batch_size=256, shuffle=True)
    
    loss_ema = 0.5
    batches_since_grow = 0
    
    for epoch in range(5):
        hybrid_model.train()
        hybrid_model.cnn.eval()
        
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            
            # Use only the output head for the current task
            pred = hybrid_model(bx)[:, task_id].unsqueeze(1)
            loss = criterion(pred, by)
            
            if replay.has_data():
                rx, ry, rt = replay.sample(128)
                rx, ry, rt = rx.to(device), ry.to(device), rt.to(device)
                
                # We must route the replay samples to their respective output heads!
                r_pred_all_heads = hybrid_model(rx)
                # Select the correct head for each replay sample
                r_pred = r_pred_all_heads[torch.arange(rx.size(0)), rt].unsqueeze(1)
                
                loss_r = criterion(r_pred, ry)
                
                grads_task = torch.autograd.grad(loss, hybrid_model.dynamic.W, retain_graph=True, allow_unused=True)[0]
                grads_replay = torch.autograd.grad(loss_r, hybrid_model.dynamic.W, retain_graph=True, allow_unused=True)[0]
                
                if grads_task is not None and grads_replay is not None:
                    # Generation 2: Update Local Per-Neuron Stress!
                    hybrid_model.dynamic.update_stress(grads_task, grads_replay)
                    
            loss.backward()
            optimizer.step()
            
            loss_ema = 0.9 * loss_ema + 0.1 * loss.item()
            # Gen 2: We check the MAX local stress among all neurons
            max_stress = hybrid_model.dynamic.get_max_neuron_stress()
            batches_since_grow += 1
            
            if max_stress > 0.3:
                # Generation 2: Localized Freeze (only freezes stressed neurons)
                n_frozen = hybrid_model.dynamic.stress_freeze(threshold=0.3)
                if n_frozen > 0:
                    hybrid_model.dynamic.grow_neuron(num_connections=15)
                    _reset_adam_for_neuron(optimizer, hybrid_model.dynamic, hybrid_model.dynamic.active_neurons - 1)
                    batches_since_grow = 0
                    print(f"  [Batch] LOCAL FREEZE: {n_frozen} conns. Grew 1. active={hybrid_model.dynamic.active_neurons} Max Stress={max_stress:.2f}")
            
            elif loss_ema > 0.2 and batches_since_grow > 20 and max_stress <= 0.3:
                hybrid_model.dynamic.grow_neuron(num_connections=15)
                _reset_adam_for_neuron(optimizer, hybrid_model.dynamic, hybrid_model.dynamic.active_neurons - 1)
                batches_since_grow = 0
                print(f"  [Batch] GROW(Loss): active={hybrid_model.dynamic.active_neurons} loss={loss_ema:.4f}")
        
        print(f"  [Epoch {epoch+1:2d}] active={hybrid_model.dynamic.active_neurons} loss={loss_ema:.4f}")
        
    replay.add_data(X_train, y_train, task_id, num_samples=200)
    
    hybrid_model.eval()
    print(f"Accuracy after {name}:")
    with torch.no_grad():
        for eval_id in range(task_id + 1):
            t_name, _, _, t_X_test, t_y_test = tasks[eval_id]
            t_X_test_d = t_X_test.to(device)
            t_y_test_d = t_y_test.to(device)
            # Route evaluation through its respective Output Head
            preds = hybrid_model(t_X_test_d)[:, eval_id].unsqueeze(1)
            acc = ((preds > 0.5).float() == t_y_test_d).float().mean().item()
            print(f"  {t_name}: {acc*100:.1f}%")
    
    print(f"  Active Neurons: {hybrid_model.dynamic.active_neurons} (trainable={hybrid_model.dynamic.n_trainable()}, frozen={hybrid_model.dynamic.n_frozen()})")

